In [ ]:
# Toda célula de um caderno marimo é uma função Python; o que ela devolve no "return" fica para as outras células.
import marimo as mo

In [ ]:
# O PAINEL: a história em português simples e, numa caixa fechada, os detalhes técnicos (lê o PIPELINE.md).
import sys as _sys
_pasta = str(mo.notebook_dir())
if _pasta not in _sys.path:
    _sys.path.insert(0, _pasta)
import painel
painel.mostrar(mo)

# Ciclo 2, etapa 9: a régua (a primeira tentativa)

**Em português:** antes de testar modelos sofisticados, medimos três jeitos simples de decidir, para saber o que
"bom" quer dizer aqui. Todo modelo que vier depois tem de ganhar destes. A régua do contrato do ciclo 2 é a **regra
do app**: o modelo tem de **achar mais** fontes do mesmo fato que ela, **sem errar mais que 3 a mais de cada 100**
que mostra.

| | Em português | Em termos técnicos |
|---|---|---|
| **Cada exemplo** | uma notícia e uma fonte que a busca do app trouxe | um par (afirmação × título) |
| **O que o computador vê** | 34 informações por exemplo (as do ciclo 1) | 34 colunas do pré-processamento base |
| **O que ele adivinha** | "é o mesmo fato?" sim ou não | alvo `mesmo`, binário |
| **Quantos "sim"** | uns 13 de cada 100; por isso os "sim" pesam mais no treino | `class_weight="balanced"` |
| **Jeito 1: o chute** | dizer sempre "não" | `DummyClassifier` |
| **Jeito 2: a regra do app** | o que o app faz hoje, sem aprender nada | a régua do contrato |
| **Jeito 3: o modelo simples** | um peso por informação, somados numa nota de 0 a 1 | regressão logística (a do ciclo 1, C=0,1) |
| **A nota de corte** | a partir de qual nota o modelo diz "sim": no simulado, no ponto em que ele erra o mesmo que a regra | `corte_com_precisao` com o piso = precisão da regra |

**Um cuidado:** o corte é escolhido no simulado e o resultado é medido no simulado, o que deixa o número um pouco
otimista, igual para todos os modelos. O número que vale sai da prova, que continua no cofre.

**O que eu espero, escrito antes de rodar:** a regra no simulado com precisão entre 80 e 88 de cada 100 e acha de 4 a
5,5 de cada 10; a logística, no corte em que erra o mesmo que a regra, acha de 6 a 8 de cada 10, com PR-AUC entre 0,85
e 0,92; com o alvo embaralhado, a PR-AUC cai para perto de 0,13 (a taxa de "mesmo"); nenhuma repetição entre estudo e
simulado; o canário abaixo de 0,80. Se a logística passar de 8 de cada 10, a primeira hipótese é cola, não vitória.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Os módulos do pipeline moram na pasta deste caderno.
if str(mo.notebook_dir()) not in sys.path:
    sys.path.insert(0, str(mo.notebook_dir()))
import guard
from dados import carregar
from medidas import MARGEM, corte_com_precisao, passa_no_contrato, precisao_e_cobertura
from preprocessamento import montar_preprocessador

# Estudo e simulado, da tabela pronta (hash conferido) e da divisão da etapa 6. A prova não sai do carregar().
estudo, simulado = carregar()

# A CHAVE DE TESTE (o Matheus roda os treinos; a sessão de IA só confere que o código não quebra): com a variável de
# ambiente ML_AMOSTRA_TESTE ligada (ex.: 200), o caderno usa poucas notícias inteiras e grava numa pasta descartável;
# nenhum número dessa rodada vale. No VS Code a variável não existe, e tudo roda com os exemplos inteiros.
AMOSTRA_TESTE = int(os.environ.get("ML_AMOSTRA_TESTE", "0"))
PASTA_SAIDA = mo.notebook_dir()
if AMOSTRA_TESTE:
    PASTA_SAIDA = Path(tempfile.mkdtemp(prefix="teste_caderno_"))

    def _poucas(tabela, alvo):
        # notícias inteiras (para não quebrar a regra dos grupos), em ordem fixa, até juntar uns `alvo` exemplos
        _guardar, _soma = [], 0
        for _g, _n in tabela.groupby("grupo").size().sort_index().items():
            _guardar.append(_g)
            _soma += _n
            if _soma >= alvo:
                break
        return tabela[tabela["grupo"].isin(_guardar)]
    estudo = _poucas(estudo, AMOSTRA_TESTE)
    simulado = _poucas(simulado, AMOSTRA_TESTE)
MESMO_DA_REGRA = ["mesmo_acontecimento", "detalhe_divergente"]

In [ ]:
# JEITO 1: o palpite "nunca é mesmo". Não precisa de modelo: é só dizer "não" para todo par.
from sklearn.dummy import DummyClassifier

# strategy="most_frequent": aprende qual classe é a mais comum no estudo ("não mesmo") e responde sempre ela.
modelo_palpite = DummyClassifier(strategy="most_frequent")
modelo_palpite.fit(estudo[["cosseno"]], estudo["mesmo"])        # a entrada é ignorada; só a contagem importa
_aponta = modelo_palpite.predict(simulado[["cosseno"]])
# A acurácia (fração de acertos) parece alta e não diz nada: acerta todos os "não" e nenhum "mesmo".
regua_palpite = {"acurácia": round(float((_aponta == simulado["mesmo"]).mean()), 3),
                 "acha de cada 10": round(10 * precisao_e_cobertura(simulado["mesmo"], _aponta)[1], 2)}
regua_palpite

In [ ]:
# JEITO 2: a regra do app, a régua do contrato. Não treina nada: a decisão dela já está na tabela.
aponta_regra = simulado["regra"].isin(MESMO_DA_REGRA).to_numpy()
precisao_regra, cobertura_regra = precisao_e_cobertura(simulado["mesmo"], aponta_regra)
regua_regra = {"precisão": round(precisao_regra, 3), "acha de cada 10": round(10 * cobertura_regra, 2),
               "apontados": int(aponta_regra.sum())}
regua_regra

In [ ]:
# JEITO 3: a regressão logística, o primeiro modelo treinado do ciclo 2. O treino, aberto linha a linha.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline

# O pré-processamento base da etapa 8 (as 34 colunas do ciclo 1, sem a informação nova nem a regra) e o modelo,
# juntos num Pipeline: o .fit abaixo ajusta os dois SÓ com o estudo, na ordem, e o app usaria o mesmo objeto.
_preprocessador = montar_preprocessador(com_regra=False, interacao=False)
modelo_logistico = Pipeline([
    ("preproc", _preprocessador),
    ("logistica", LogisticRegression(
        # C: o inverso da força do "freio" que impede pesos exagerados. 0,1 foi o escolhido na etapa 11 do ciclo 1;
        # o ajuste fino dele, neste ciclo, é a etapa 11 de novo.
        C=0.1,
        # class_weight="balanced": cada "mesmo" pesa mais no treino, na proporção inversa da sua frequência
        # (uns 13 de cada 100). Sem isso, o modelo aprenderia que dizer "não" quase sempre já acerta muito.
        class_weight="balanced",
        # max_iter: quantas voltas o otimizador pode dar até convergir; 3000 sobra para 34 colunas.
        max_iter=3000,
    )),
])
entradas = _preprocessador.entradas

# O TREINO: aprende os pesos olhando só os pares do estudo (entradas e alvo).
modelo_logistico.fit(estudo[entradas], estudo["mesmo"])

# A NOTA no simulado: predict_proba devolve [chance de "não", chance de "mesmo"]; [:, 1] pega a segunda coluna.
nota_logistica = modelo_logistico.predict_proba(simulado[entradas])[:, 1]
# O CORTE do contrato: o mais frouxo em que a logística erra no máximo o mesmo que a regra no simulado.
corte_logistica = corte_com_precisao(simulado["mesmo"], nota_logistica, piso=precisao_regra)
aponta_logistica = nota_logistica >= corte_logistica["corte"]
# PR-AUC: a qualidade da ORDEM das notas, em todos os cortes de uma vez (o acaso fica na taxa de "mesmo").
pr_auc_logistica = float(average_precision_score(simulado["mesmo"], nota_logistica))
{"corte": round(corte_logistica["corte"], 4), "precisão": round(corte_logistica["precisao"], 3),
 "acha de cada 10": round(10 * corte_logistica["cobertura"], 2), "apontados": corte_logistica["apontados"],
 "PR-AUC": round(pr_auc_logistica, 3)}

In [ ]:
# O CONTRATO NO SIMULADO (ensaio; o que vale é a prova): as duas exigências, com os intervalos sorteando notícias.
_y = simulado["mesmo"].to_numpy()
contrato_simulado = passa_no_contrato(_y, aponta_logistica, aponta_regra, simulado["grupo"])

# AS FATIAS do contrato: por tipo de busca; sem os desempatados; sem os "mesmo" de fonte velha.
_fatias = {
    "todo o simulado": simulado.index == simulado.index,
    "a regra comparou com cuidado": (simulado["comparou"] == 1).to_numpy(),
    "só relevância": (simulado["comparou"] == 0).to_numpy(),
    "sem os desempatados (maioria de três)": (simulado["origem"] != "maioria de três").to_numpy(),
    "sem os \"mesmo\" de fonte velha": ~simulado["mesmo_de_fonte_velha"].to_numpy(),
}
_linhas = []
for _nome, _m in _fatias.items():
    _pr, _cr = precisao_e_cobertura(_y[_m], aponta_regra[_m])
    _pl, _cl = precisao_e_cobertura(_y[_m], aponta_logistica[_m])
    _linhas.append({"fatia": _nome, "fontes": int(_m.sum()), "do mesmo fato": int(_y[_m].sum()),
                    "regra: acha de cada 10": round(10 * _cr, 2), "regra: precisão": round(_pr, 3),
                    "modelo: acha de cada 10": round(10 * _cl, 2), "modelo: precisão": round(_pl, 3)})
regua_por_fatia = pd.DataFrame(_linhas).set_index("fatia")
_c, _p = contrato_simulado["cobertura"], contrato_simulado["precisao"]
resumo_contrato = {
    "achar mais: modelo menos regra (de cada 10)": f"{10 * _c['diferenca']:+.2f} [{10 * _c['de']:+.2f}; {10 * _c['ate']:+.2f}]",
    "não errar demais: precisão modelo menos regra (pontos)": f"{100 * _p['diferenca']:+.1f} [{100 * _p['de']:+.1f}; {100 * _p['ate']:+.1f}] (margem -{100 * MARGEM:.0f})",
    "passaria no contrato (ensaio no simulado)": contrato_simulado["passa"],
}
(resumo_contrato, regua_por_fatia)

In [ ]:
# FIGURA DA ETAPA 9: a curva precisão × cobertura da logística no simulado, com a regra e a linha da regra.
from sklearn.metrics import precision_recall_curve
_prec, _cob, _ = precision_recall_curve(simulado["mesmo"], nota_logistica)
_pr, _cr = precisao_e_cobertura(simulado["mesmo"], aponta_regra)
_fig, _ax = plt.subplots(figsize=(6.5, 4.2))
_ax.plot(_cob, _prec, label="regressão logística (todos os cortes)")
_ax.axhline(_pr, linestyle="--", color="gray", linewidth=1, label="precisão da regra (o corte do contrato)")
_ax.scatter([_cr], [_pr], color="black", zorder=3, label="regra do app")
_ax.scatter([corte_logistica["cobertura"]], [corte_logistica["precisao"]], color="red", zorder=3, label="logística no corte")
_ax.set_xlabel("cobertura (das fontes do mesmo fato, quantas aponta)")
_ax.set_ylabel("precisão (do que aponta, quanto é o mesmo fato)")
_ax.set_title(f"A régua no simulado ({simulado['grupo'].nunique()} notícias)")
_ax.legend(fontsize=7, loc="lower left")
_d = contrato_simulado["cobertura"]
_explicacao = (
    f"No simulado, a regra do app acha {10 * _cr:.1f} de cada 10 fontes do mesmo fato, com precisão de {_pr:.0%}; a "
    f"logística, no corte em que erra o mesmo que a regra, acha {10 * corte_logistica['cobertura']:.1f} (precisão "
    f"{corte_logistica['precisao']:.0%}). Diferença: {10 * _d['diferenca']:+.1f} de cada 10 (faixa de {10 * _d['de']:+.1f} a "
    f"{10 * _d['ate']:+.1f}, sorteando notícias). É a régua do ciclo 2: todo modelo seguinte precisa ganhar dela."
)
figura_regua = guard.fig(9, "regua_no_simulado", _fig, _explicacao, state_dir=PASTA_SAIDA)
mo.vstack([mo.image(src=str(figura_regua), width=560), mo.md(_explicacao)])

In [ ]:
# A MESMA RÉGUA, NO IDIOMA DO PRODUTO: "de cada 100 fontes que a busca traz...". Números do simulado, calculados aqui.
_y = simulado["mesmo"].to_numpy()
_por_100 = 100 / len(simulado)
_verdadeiras = _y.sum() * _por_100

def _conta(aponta):
    return (aponta.sum() * _por_100, (aponta & _y).sum() * _por_100, (aponta & ~_y).sum() * _por_100)

_r, _m = _conta(aponta_regra), _conta(aponta_logistica)
mo.md(f"""
## O que isso quer dizer no app, em português simples

De cada **100 fontes** que a busca traz, **{_verdadeiras:.1f}** são o mesmo fato (no simulado). Quem decide mostra:

| Quem decide | Mostra como "mesmo fato" | Certas | Erradas |
|---|---|---|---|
| **A regra do app, hoje** | {_r[0]:.1f} | {_r[1]:.1f} | {_r[2]:.1f} |
| **O modelo simples** | {_m[0]:.1f} | {_m[1]:.1f} | {_m[2]:.1f} |

O corte do modelo foi posto no ponto em que ele erra **na mesma proporção** que a regra; a diferença está em quantas
certas cada um mostra. É ensaio: o número que vale sai da prova.
""")

## Os testes de cola, antes de acreditar em qualquer número

**Em português:** resultado bom é o primeiro suspeito em machine learning. Os quatro testes do ciclo 1, de novo:

| Teste | Em português | Se fosse cola... |
|---|---|---|
| 1. Respostas trocadas | ensinamos com as respostas embaralhadas: sem resposta certa, o modelo deveria ficar perdido | ...ele continuaria indo bem |
| 2. Só a semelhança | quanto a semelhança que o app já calcula ordena sozinha, sem treino | (mostra de onde vem o ganho) |
| 3. Exemplo repetido | alguma notícia ou par do simulado também está no estudo? | ...ele estaria lembrando, não aprendendo |
| 4. O canário | um exercício inventado cujo melhor resultado honesto é 80% | ...passaria de 80% |

In [ ]:
from sklearn.linear_model import LogisticRegression as _Logistica
from sklearn.metrics import average_precision_score as _pr_auc
from sklearn.pipeline import Pipeline as _Pipeline

_y_sim = simulado["mesmo"].to_numpy()

# TESTE 1: o mesmo modelo treinado com o alvo EMBARALHADO (rng.permutation sorteia uma ordem nova), 5 vezes.
_embaralhados = []
for _semente in range(5):
    _rng = np.random.default_rng(_semente)
    _modelo_embaralhado = _Pipeline([("preproc", montar_preprocessador(com_regra=False, interacao=False)),
                                     ("logistica", _Logistica(C=0.1, class_weight="balanced", max_iter=3000))])
    _modelo_embaralhado.fit(estudo[entradas], _rng.permutation(estudo["mesmo"].to_numpy()))
    _embaralhados.append(round(float(_pr_auc(_y_sim, _modelo_embaralhado.predict_proba(simulado[entradas])[:, 1])), 3))

# TESTE 3: repetição entre estudo e simulado (texto normalizado: minúsculas, espaços únicos).
_norm = lambda s: " ".join(str(s).lower().split())
_af_estudo = set(estudo["afirmacao"].map(_norm))
_par_estudo = set(zip(estudo["afirmacao"].map(_norm), estudo["titulo"].map(_norm)))
_af_repetidas = sum(a in _af_estudo for a in set(simulado["afirmacao"].map(_norm)))
_pares_repetidos = sum(p in _par_estudo for p in zip(simulado["afirmacao"].map(_norm), simulado["titulo"].map(_norm)))

testes_de_cola = {
    "taxa de 'mesmo' no simulado (o acaso da PR-AUC)": round(float(_y_sim.mean()), 3),
    "1. PR-AUC com o alvo embaralhado (5 sorteios)": _embaralhados,
    "2. semelhança sozinha: PR-AUC": round(float(_pr_auc(_y_sim, simulado["cosseno"])), 3),
    "3. afirmações do simulado repetidas no estudo": _af_repetidas,
    "3. pares do simulado repetidos no estudo": _pares_repetidos,
}
assert _af_repetidas == 0 and _pares_repetidos == 0, "repetição entre estudo e simulado"
testes_de_cola

In [ ]:
# TESTE 4: o canário do pipeline: dado sintético em que nenhum modelo honesto passa de 80% de acurácia fora do treino.
from sklearn.ensemble import HistGradientBoostingClassifier as _Arvores
from sklearn.linear_model import LogisticRegression as _Logistica2
from sklearn.pipeline import Pipeline as _Pipeline2
from sklearn.preprocessing import StandardScaler as _Escala
import mlpipeline_canary as _canario

_linhas = np.array(_canario.make_rows(n=4000, seed=7), dtype=float)
_X, _y = _linhas[:, :-1], _linhas[:, -1].astype(int)
_ordem = np.random.default_rng(20261008).permutation(len(_y))
_tr, _fora = _ordem[:3000], _ordem[3000:]
teste_canario = {}
for _nome, _modelo in [
    ("logística", _Pipeline2([("preproc_escala", _Escala()), ("logistica", _Logistica2(max_iter=2000))])),
    ("árvores em sequência", _Arvores(random_state=0)),
]:
    _modelo.fit(_X[_tr], _y[_tr])
    _acc = float((_modelo.predict(_X[_fora]) == _y[_fora]).mean())
    teste_canario[_nome] = f"{_acc:.3f} ({'VAZOU' if _canario.verdict(_acc, len(_fora))[0] else 'não vazou'}; teto 0,80)"
teste_canario

In [ ]:
# OS RESULTADOS EM ARQUIVO: tudo o que esta etapa mediu, num JSON que a sessão de IA lê direto do disco (sem print).
# Pedido do Matheus em 08/10/2026. No teste pequeno, vai para a pasta descartável.
import json as _json
_resultado = {
    "etapa": 9, "o que é": "a régua do ciclo 2 no simulado",
    "palpite 'nunca é mesmo'": regua_palpite,
    "regra do app": regua_regra,
    "logística (C=0,1) no corte do contrato": {"corte": corte_logistica["corte"], "precisão": corte_logistica["precisao"],
                                                "acha de cada 10": 10 * corte_logistica["cobertura"],
                                                "apontados": corte_logistica["apontados"], "PR-AUC": pr_auc_logistica},
    "contrato no simulado (ensaio)": contrato_simulado,
    "por fatia": regua_por_fatia.to_dict(orient="index"),
    "testes de cola": testes_de_cola,
    "canário": teste_canario,
}
_pasta = PASTA_SAIDA / "resultados"
_pasta.mkdir(exist_ok=True)
(_pasta / "09_regua.json").write_text(_json.dumps(_resultado, indent=1, ensure_ascii=False, default=float) + "\n",
                                     encoding="utf-8", newline="\n")
f"resultados gravados em {_pasta / '09_regua.json'}"

# Ciclo 2, etapa 10: os candidatos contra a régua

**Em português:** a régua é o modelo simples da etapa 9 (acha 7,5 de cada 10 no simulado). Agora testamos os jeitos
que o guia de seleção de modelos e as etapas anteriores apontaram, **todos com a mesma régua**: o corte de cada um é posto no
ponto em que ele erra o mesmo que a regra do app, e a pergunta é quantas fontes do mesmo fato cada um acha. Só conta
como **vitória** quem fica acima da régua com a faixa de incerteza toda acima; se a faixa cruza a régua, é **empate**,
e empate fica com o mais simples.

| # | Candidato | Em português |
|---|---|---|
| 0 | régua | o modelo simples da etapa 9 |
| 1A | + semelhança × idade | a informação nova da etapa 7 |
| 1B | + resposta da regra | o que o app concluiu, como mais uma informação |
| 1C | + as duas | 1A e 1B juntas |
| 2 | sem a rodada 1 no estudo | a sua decisão do contrato: o simulado diz se a rodada 1 atrapalha |
| 3A | árvores em sequência | o padrão forte do guia; cruzam informações sozinhas |
| 3B | simples com todos os cruzamentos | cada par de informações multiplicado, com freio forte |
| 4 | árvores que conhecem o site (CatBoost) | o guia prefere para coluna de muitos valores |
| 5 | leitor de palavras | a régua de texto do guia |
| 6A | modelo de linguagem sozinho | lê notícia e título juntos (treinado por você na RTX) |
| 6B | simples + a nota do modelo de linguagem | a nota dele como mais uma informação, sem cola (notas fora da dobra) |
| 7 | app híbrido | a regra decide onde ela compara com cuidado; o modelo simples, no resto |

**O que eu espero, escrito antes de rodar (diferença para a régua, de cada 10 fontes do mesmo fato):** 1A de 0 a +0,6;
1B de -0,3 a +0,3; 1C de 0 a +0,6; 2 de -0,3 a +0,3; 3A de -0,8 a +0,4; 3B de -0,5 a +0,3; 4 de -0,5 a +0,3; 5 acha
menos de 2 de cada 10; 6A acha de 4 a 6; 6B de 0 a +0,5; 7 de -0,3 a +0,4, com precisão pelo menos a da regra. Se
algum passar de +1,0, a primeira hipótese é cola.

In [ ]:
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.metrics import average_precision_score as _pr_auc10
from sklearn.pipeline import Pipeline as _Pip

def avaliar(nota):
    """A medida do contrato para uma nota: o corte em que erra o mesmo que a regra, e quanto acha nesse corte."""
    nota = np.asarray(nota, dtype=float)
    _c = corte_com_precisao(simulado["mesmo"], nota, piso=precisao_regra)
    return {"corte": _c["corte"], "precisao": _c["precisao"], "cobertura": _c["cobertura"],
            "pr_auc": float(_pr_auc10(simulado["mesmo"], nota)), "aponta": nota >= _c["corte"]}

def logistica(com_regra=False, interacao=False, tabela=estudo):
    """A régua com as opções do pré-processamento: mesma configuração (C=0,1, classes equilibradas)."""
    _prep = montar_preprocessador(com_regra=com_regra, interacao=interacao)
    _modelo = _Pip([("preproc", _prep), ("logistica", _LR(C=0.1, class_weight="balanced", max_iter=3000))])
    _modelo.fit(tabela[_prep.entradas], tabela["mesmo"])                 # aprende só com o estudo (ou parte dele)
    return _modelo.predict_proba(simulado[_prep.entradas])[:, 1]         # a nota no simulado

resultados = {"0. régua": avaliar(nota_logistica)}
resultados["1A. + semelhança × idade"] = avaliar(logistica(interacao=True))
resultados["1B. + resposta da regra"] = avaliar(logistica(com_regra=True))
resultados["1C. + as duas"] = avaliar(logistica(com_regra=True, interacao=True))
# 2: o mesmo modelo simples, aprendendo sem a rodada 1 (a outra versão do app); medido no mesmo simulado.
resultados["2. sem a rodada 1 no estudo"] = avaliar(logistica(tabela=estudo[estudo["rodada"] != 1]))
{k: round(10 * v["cobertura"], 2) for k, v in resultados.items()}

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression as _LR3
from sklearn.pipeline import Pipeline as _P3
from sklearn.preprocessing import PolynomialFeatures

# 3A: árvores em sequência. Cada árvore nova faz perguntas ("semelhança acima de 0,7?", "fonte de hoje?") e corrige o
# que as anteriores erraram; a soma de todas dá a nota.
modelo_arvores = _P3([
    ("preproc", montar_preprocessador(com_regra=False)),
    ("arvores", HistGradientBoostingClassifier(
        max_depth=3,               # árvores rasas: no máximo 3 perguntas seguidas (o guia: "shallow trees")
        learning_rate=0.05,        # cada árvore corrige só um pouco: aprende devagar e decora menos
        max_iter=1000,             # no máximo 1.000 árvores...
        early_stopping=True,       # ...mas para antes, quando parar de melhorar
        validation_fraction=0.15,  # medido em 15% do ESTUDO separados por ela (o simulado não é usado aqui)
        n_iter_no_change=30,       # "parar de melhorar" = 30 árvores seguidas sem ganho
        l2_regularization=1.0,     # freio contra respostas extremas
        class_weight="balanced",   # mais atenção aos "mesmo", como na régua
        random_state=0,            # fixa os sorteios internos: rodar de novo dá o mesmo número
    )),
])
modelo_arvores.fit(estudo[entradas], estudo["mesmo"])
resultados["3A. árvores em sequência"] = avaliar(modelo_arvores.predict_proba(simulado[entradas])[:, 1])

# 3B: a logística vendo também cada PAR de informações multiplicadas; 34 colunas viram umas 600, por isso freio forte.
modelo_cruzado = _P3([
    ("preproc", montar_preprocessador(com_regra=False)),
    ("preproc_cruzamentos", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ("logistica", _LR3(C=0.01, class_weight="balanced", max_iter=5000)),
])
modelo_cruzado.fit(estudo[entradas], estudo["mesmo"])
resultados["3B. simples com todos os cruzamentos"] = avaliar(modelo_cruzado.predict_proba(simulado[entradas])[:, 1])
round(10 * resultados["3A. árvores em sequência"]["cobertura"], 2), round(10 * resultados["3B. simples com todos os cruzamentos"]["cobertura"], 2)

In [ ]:
from catboost import CatBoostClassifier

# 4: árvores do CatBoost, que sabem usar coluna de muitos valores (o site) sem criar uma coluna por site. Recebem as
# informações cruas (o CatBoost faz o próprio tratamento das categorias, aprendido só no estudo).
_colunas4 = entradas + ["veiculo"]
_categoricas4 = ["quem", "acao", "assunto", "quando", "buscador", "tipo_afirmacao", "veiculo"]
modelo_catboost = CatBoostClassifier(
    iterations=600,                  # quantas árvores (sem parada antecipada: o simulado não escolhe isso aqui)
    depth=4,                         # árvores de até 4 perguntas
    learning_rate=0.05,
    auto_class_weights="Balanced",   # mais atenção aos "mesmo"
    random_seed=0,
    verbose=0,                       # não imprime nada durante o treino
    allow_writing_files=False,       # não grava a pasta catboost_info/
)
modelo_catboost.fit(estudo[_colunas4].fillna({"veiculo": "sem"}), estudo["mesmo"], cat_features=_categoricas4)
resultados["4. árvores que conhecem o site"] = avaliar(
    modelo_catboost.predict_proba(simulado[_colunas4].fillna({"veiculo": "sem"}))[:, 1])
round(10 * resultados["4. árvores que conhecem o site"]["cobertura"], 2)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as _LR5

# 5: o leitor de palavras. Só texto: nada de semelhança, idade ou regra.
# 1) O vocabulário e o "peso" de cada palavra (raras pesam mais) são aprendidos SÓ nos textos do estudo.
vec_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),       # palavras soltas e pares de palavras seguidas ("supremo tribunal")
    min_df=2,                 # ignora o que aparece em menos de 2 textos
    sublinear_tf=True,        # palavra repetida conta menos que o dobro
    strip_accents="unicode",  # "decisão" e "decisao" viram a mesma palavra
)
vec_tfidf.fit(pd.concat([estudo["afirmacao"], estudo["titulo"]]))

def _palavras_em_comum(tabela):
    # 2) Cada texto vira uma lista de pesos por palavra; multiplicar as duas listas zera tudo que não está nos dois
    #    textos. Sobram só as palavras EM COMUM, cada uma com o seu peso: é isso que o modelo lê.
    return vec_tfidf.transform(tabela["afirmacao"]).multiply(vec_tfidf.transform(tabela["titulo"]))

# 3) A logística aprende quais palavras em comum indicam "mesmo fato".
modelo_palavras = _LR5(C=1.0, class_weight="balanced", max_iter=5000)
modelo_palavras.fit(_palavras_em_comum(estudo), estudo["mesmo"])
resultados["5. leitor de palavras"] = avaliar(modelo_palavras.predict_proba(_palavras_em_comum(simulado))[:, 1])
round(10 * resultados["5. leitor de palavras"]["cobertura"], 2)

In [ ]:
from sklearn.linear_model import LogisticRegression as _LR6
from sklearn.pipeline import Pipeline as _P6

# 6A e 6B: as notas do modelo de linguagem, gravadas pelo treinar_cross_encoder.py (rodado pelo Matheus na RTX).
_arq_sim = mo.notebook_dir() / "notas_cross_encoder_simulado.csv"
_arq_dobras = mo.notebook_dir() / "notas_cross_encoder_estudo_dobras.csv"
if not (_arq_sim.is_file() and _arq_dobras.is_file()):
    aviso_linguagem = "Ainda sem as notas do modelo de linguagem: rode o treinar_cross_encoder.py (dobras e sozinho) na RTX."
else:
    # 6A: a nota dele no simulado, sozinha. reindex garante a ordem dos pares do simulado (no teste pequeno, os que existem).
    _nota_sim = pd.read_csv(_arq_sim, index_col="id")["nota"].reindex(simulado.index)
    _nota_est = pd.read_csv(_arq_dobras, index_col="id")["nota"].reindex(estudo.index)
    if _nota_sim.isna().any() or _nota_est.isna().any():
        aviso_linguagem = "As notas do modelo de linguagem não cobrem todos os pares (teste pequeno?): 6A e 6B ficam de fora."
    else:
        resultados["6A. modelo de linguagem sozinho"] = avaliar(_nota_sim.to_numpy())
        # 6B: a régua com uma informação a mais, a nota do texto. No estudo, a nota "fora da dobra" (de um modelo que
        # não viu o par); no simulado, a do modelo treinado com o estudo inteiro (que também não viu o simulado).
        _prep6 = montar_preprocessador(com_regra=False, extras=("nota_texto",))
        modelo_com_texto = _P6([("preproc", _prep6),
                                ("logistica", _LR6(C=0.1, class_weight="balanced", max_iter=3000))])
        modelo_com_texto.fit(estudo.assign(nota_texto=_nota_est)[_prep6.entradas], estudo["mesmo"])
        resultados["6B. simples + nota do texto"] = avaliar(
            modelo_com_texto.predict_proba(simulado.assign(nota_texto=_nota_sim)[_prep6.entradas])[:, 1])
        aviso_linguagem = "6A e 6B prontos."
aviso_linguagem

In [ ]:
# 7: o APP HÍBRIDO. Onde a regra comparou com cuidado, vale a regra; no resto, a régua (o modelo simples), com o corte
# mais frouxo em que o conjunto todo ainda erra no máximo o mesmo que a regra.
_y = simulado["mesmo"].to_numpy()
_comparou = (simulado["comparou"] == 1).to_numpy()
_regra_diz = simulado["regra"].isin(MESMO_DA_REGRA).to_numpy()
_melhor = None
for _c in np.unique(nota_logistica[~_comparou]):            # do corte mais baixo ao mais alto
    _aponta = np.where(_comparou, _regra_diz, nota_logistica >= _c)
    _p, _cob = precisao_e_cobertura(_y, _aponta)
    if _p >= precisao_regra:                                 # o primeiro que respeita a precisão da regra é o mais frouxo
        _melhor = {"corte": float(_c), "precisao": _p, "cobertura": _cob, "pr_auc": float("nan"), "aponta": _aponta}
        break
resultados["7. app híbrido"] = _melhor or {"corte": float("inf"), "precisao": float("nan"), "cobertura": 0.0,
                                           "pr_auc": float("nan"), "aponta": np.zeros(len(_y), dtype=bool)}
round(10 * resultados["7. app híbrido"]["cobertura"], 2)

In [ ]:
import json as _json10
from medidas import intervalo_da_diferenca as _iv_dif

# O PLACAR: cada candidato contra a régua, com a faixa da diferença (sorteando notícias do simulado).
_y10 = simulado["mesmo"].to_numpy()
_regua = resultados["0. régua"]
_linhas10 = []
for _nome, _r in resultados.items():
    _iv = (_iv_dif(_y10, _r["aponta"], _regua["aponta"], simulado["grupo"]) if _nome != "0. régua"
           else {"diferenca": 0.0, "de": 0.0, "ate": 0.0})
    _linhas10.append({
        "candidato": _nome,
        "acha de cada 10": round(10 * _r["cobertura"], 2),
        "precisão": round(_r["precisao"], 3),
        "PR-AUC": round(_r["pr_auc"], 3),
        "contra a régua (de cada 10)": round(10 * _iv["diferenca"], 2),
        "faixa de": round(10 * _iv["de"], 2),
        "até": round(10 * _iv["ate"], 2),
        "resultado": "régua" if _nome == "0. régua" else ("ganha" if _iv["de"] > 0 else ("perde" if _iv["ate"] < 0 else "empate")),
    })
placar = pd.DataFrame(_linhas10).set_index("candidato")

_fig, _ax = plt.subplots(figsize=(8, 4.4))
_ax.barh(placar.index, placar["acha de cada 10"])
_baixo = placar["contra a régua (de cada 10)"] - placar["faixa de"]
_cima = placar["até"] - placar["contra a régua (de cada 10)"]
_ax.errorbar(placar["acha de cada 10"], range(len(placar)), xerr=[_baixo, _cima], fmt="none", ecolor="black", capsize=3)
_ax.axvline(placar.loc["0. régua", "acha de cada 10"], linestyle="--", color="gray", linewidth=1)
_ax.invert_yaxis()
_ax.tick_params(axis="y", labelsize=7)
_ax.set_xlabel("de cada 10 fontes do mesmo fato, quantas acha (errando o mesmo que a regra do app)")
_ax.set_title("Ciclo 2, etapa 10: os candidatos contra a régua, no simulado")
_melhor = placar["acha de cada 10"].idxmax()
_ganham = list(placar.index[placar["resultado"] == "ganha"])
_explicacao = (
    f"No simulado, a régua acha {placar.loc['0. régua', 'acha de cada 10']} de cada 10 fontes do mesmo fato; o que mais "
    f"acha é '{_melhor}', com {placar.loc[_melhor, 'acha de cada 10']}. "
    + (f"Ganham da régua com a faixa toda acima: {', '.join(_ganham)}. " if _ganham else "Nenhum ganha da régua com a faixa toda acima. ")
    + "A barra preta é a faixa de incerteza da diferença para a régua (linha tracejada): se cruza a linha, é empate."
)
figura_placar = guard.fig(10, "candidatos_contra_a_regua", _fig, _explicacao, state_dir=PASTA_SAIDA)

# OS RESULTADOS EM ARQUIVO, para a sessão de IA ler do disco.
_pasta = PASTA_SAIDA / "resultados"
_pasta.mkdir(exist_ok=True)
(_pasta / "10_candidatos.json").write_text(
    _json10.dumps({"etapa": 10, "placar": placar.to_dict(orient="index"),
                   "cortes": {k: v["corte"] for k, v in resultados.items()}},
                  indent=1, ensure_ascii=False, default=float) + "\n", encoding="utf-8", newline="\n")
mo.vstack([mo.image(src=str(figura_placar), width=720), mo.md(_explicacao), placar])

# Ciclo 2, etapa 11: afinar o escolhido (6B, o simples com a nota do modelo de linguagem)

**Em português:** o 6B tem "botões" na posição de fábrica (a mesma da régua). Vamos testar várias posições e ver se
alguma acha mais fontes do mesmo fato, errando o mesmo que a regra do app. O cuidado é **onde** testar: se a escolha
olhar o simulado, ele deixa de ser um ensaio justo (seria "estudar para o simulado", e o 6B já foi escolhido nele
entre 11). Então a escolha acontece **dentro do estudo**: o estudo é dividido em 5 partes por notícia inteira; o
modelo aprende com 4 e é medido na quinta, 5 vezes. O simulado só confere no fim.

| Botão | Em português | Posições testadas | Em termos técnicos |
|---|---|---|---|
| **O quanto ele se "segura"** | se segura pouco, decora o estudo; se segura muito, fica "bobo" | 8, de muito a pouco | regularização `C` de 0,003 a 10 (fábrica: 0,1) |
| **Peso dos exemplos duvidosos** | os 167 desempatados pelo Gemini valem 1 voto, meio voto ou nada | 3 | `sample_weight` da "maioria de três" |
| **Como a nota do texto entra** | como sai do modelo de linguagem (quase sempre perto de 0 ou de 1), ou "esticada", para separar 0,98 de 0,999 | 2 | a probabilidade ou o seu logit |

| Como medimos | Em português | Em termos técnicos |
|---|---|---|
| **Onde** | só a parte rodada 3 do estudo (6.536 pares), a mesma época do simulado | população da medida |
| **A régua dentro do estudo** | o modelo pode errar o mesmo que a regra do app erra **ali** | piso = precisão da regra no estudo, rodada 3 |
| **A nota de cada par** | de um modelo que nunca viu o par (e a nota do texto também veio de um que não o viu) | validação cruzada por grupo, notas fora da dobra |

**A regra de escolha, escrita antes de rodar:** fica a combinação que mais acha; se a de fábrica (C=0,1, peso 1, nota
como sai) estiver a até 0,1 de cada 10 da melhor, fica a de fábrica (mexer sem ganho é risco sem prêmio). Senão,
entre as que empatam com a melhor, a mais simples: peso 1 antes de 0,5 antes de 0; nota como sai antes de esticada;
C menor (se segura mais).

**O que eu espero, escrito antes de rodar:** os botões mudam pouco: C até 0,3 de cada 10; o peso dos duvidosos até
0,2; a nota esticada de 0 a +0,4 (a logística lida mal com nota espremida perto de 0 e 1). No simulado, com o corte
do contrato, o afinado fica a ±0,3 do de fábrica (empate). Com o corte escolhido no estudo e levado ao simulado, a
precisão pode sair de 78% a 92%, porque a nota do texto no estudo vem de modelos que aprenderam com 4/5 dele e a do
simulado de um que aprendeu com tudo: as notas podem não estar na mesma escala. Se o afinado ganhar mais de +0,5
no simulado, a primeira hipótese é sorte da escolha, não vitória.

**Quanto demora:** 48 combinações × 5 partes = 240 treinos pequenos, de 1 a 3 minutos.

In [ ]:
from sklearn.linear_model import LogisticRegression as _LR11
from sklearn.model_selection import GroupKFold as _GKF11
from sklearn.pipeline import Pipeline as _P11

# A nota do texto no estudo, "fora da dobra" (gravada pelo treinar_cross_encoder.py dobras, rodado por você).
# reindex: a ordem dos pares do estudo (no teste pequeno, só os que existem).
nota_texto_estudo = pd.read_csv(mo.notebook_dir() / "notas_cross_encoder_estudo_dobras.csv",
                                index_col="id")["nota"].reindex(estudo.index).to_numpy()

def nota_do_jeito(nota, jeito):
    """'como sai': a probabilidade do modelo de linguagem. 'esticada': o logit, log(p / (1 - p)), que abre o que
    está espremido perto de 0 e de 1 (0,98 vira 3,9; 0,999 vira 6,9). O clip evita log de zero."""
    if jeito == "como sai":
        return nota
    _p = np.clip(nota, 1e-6, 1 - 1e-6)
    return np.log(_p / (1 - _p))

# As posições testadas de cada botão.
GRADE_C_6B = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]   # C pequeno = se segura MAIS
PESOS_6B = [1.0, 0.5, 0.0]                                     # quanto vale um exemplo desempatado pelo Gemini
JEITOS_6B = ["como sai", "esticada"]                           # como a nota do texto entra
FABRICA_6B = (0.1, 1.0, "como sai")                            # a posição da etapa 10

_prep11 = montar_preprocessador(com_regra=False, extras=("nota_texto",))
entradas_6b = _prep11.entradas                                 # as 34 informações da régua + a nota do texto
_y = estudo["mesmo"].to_numpy()
_r3 = (estudo["rodada"] == 3).to_numpy()                       # a medida usa só a rodada 3 (a época do simulado)
_eh_maioria = (estudo["origem"] == "maioria de três").to_numpy()

# A régua dentro do estudo: a precisão da regra do app na parte rodada 3 do estudo.
precisao_regra_estudo, _ = precisao_e_cobertura(_y[_r3], estudo["regra"].isin(MESMO_DA_REGRA).to_numpy()[_r3])

# As 5 partes do estudo, por notícia inteira. min(5, ...): no teste pequeno pode haver menos notícias.
_dobras = list(_GKF11(n_splits=min(5, estudo["grupo"].nunique())).split(estudo, groups=estudo["grupo"]))

def notas_fora_da_dobra_6b(C, peso, jeito):
    """Para cada par do estudo, a nota de um 6B que NÃO o viu (treinado nas outras 4 partes)."""
    _tabela = estudo.assign(nota_texto=nota_do_jeito(nota_texto_estudo, jeito))
    _peso = np.where(_eh_maioria, peso, 1.0)                 # o peso de cada exemplo no treino
    _notas = np.zeros(len(estudo))
    for _dentro, _fora in _dobras:
        _m = _P11([("preproc", montar_preprocessador(com_regra=False, extras=("nota_texto",))),
                   ("logistica", _LR11(C=C, class_weight="balanced", max_iter=5000))])
        # logistica__sample_weight: o peso de cada exemplo vai direto para o passo "logistica" do Pipeline.
        _m.fit(_tabela.iloc[_dentro][entradas_6b], _y[_dentro], logistica__sample_weight=_peso[_dentro])
        _notas[_fora] = _m.predict_proba(_tabela.iloc[_fora][entradas_6b])[:, 1]
    return _notas

# O TREINO DA ETAPA 11: as 48 combinações, cada uma medida fora da dobra na parte rodada 3 do estudo.
_linhas = []
notas_por_config_6b = {}
for _C in GRADE_C_6B:
    for _peso in PESOS_6B:
        for _jeito in JEITOS_6B:
            _n = notas_fora_da_dobra_6b(_C, _peso, _jeito)
            notas_por_config_6b[(_C, _peso, _jeito)] = _n
            _c = corte_com_precisao(_y[_r3], _n[_r3], precisao_regra_estudo)
            _linhas.append({"C": _C, "peso dos duvidosos": _peso, "nota do texto": _jeito,
                            "acha, de cada 10": round(10 * _c["cobertura"], 2), "precisão": round(_c["precisao"], 3),
                            "corte": _c["corte"]})
grade_6b = pd.DataFrame(_linhas)

# A ESCOLHA, pela regra escrita antes.
_melhor = grade_6b["acha, de cada 10"].max()
_fab = grade_6b[(grade_6b["C"] == FABRICA_6B[0]) & (grade_6b["peso dos duvidosos"] == FABRICA_6B[1])
                & (grade_6b["nota do texto"] == FABRICA_6B[2])].iloc[0]
if _fab["acha, de cada 10"] >= _melhor - 0.1:
    config_6b = FABRICA_6B
else:
    _emp = grade_6b[grade_6b["acha, de cada 10"] >= _melhor - 0.1].assign(
        _sai=lambda d: d["nota do texto"] != "como sai")          # False (como sai) vem antes na ordenação
    _e = _emp.sort_values(["peso dos duvidosos", "_sai", "C"], ascending=[False, True, True], kind="stable").iloc[0]
    config_6b = (float(_e["C"]), float(_e["peso dos duvidosos"]), str(_e["nota do texto"]))
grade_6b

In [ ]:
from sklearn.linear_model import LogisticRegression as _LR11b
from sklearn.pipeline import Pipeline as _P11b
from medidas import intervalo_da_diferenca as _iv11

# A nota do texto no simulado: a do modelo de linguagem treinado com o estudo inteiro (que nunca viu o simulado).
_nota_sim = pd.read_csv(mo.notebook_dir() / "notas_cross_encoder_simulado.csv",
                        index_col="id")["nota"].reindex(simulado.index).to_numpy()
_y_e = estudo["mesmo"].to_numpy()
_r3 = (estudo["rodada"] == 3).to_numpy()
_y_s = simulado["mesmo"].to_numpy()

def treinar_6b(C, peso, jeito):
    """O 6B com uma configuração, treinado com TODO o estudo; devolve o modelo e a nota no simulado."""
    _peso = np.where((estudo["origem"] == "maioria de três").to_numpy(), peso, 1.0)
    _m = _P11b([("preproc", montar_preprocessador(com_regra=False, extras=("nota_texto",))),
                ("logistica", _LR11b(C=C, class_weight="balanced", max_iter=5000))])
    _m.fit(estudo.assign(nota_texto=nota_do_jeito(nota_texto_estudo, jeito))[entradas_6b], _y_e,
           logistica__sample_weight=_peso)
    return _m, _m.predict_proba(simulado.assign(nota_texto=nota_do_jeito(_nota_sim, jeito))[entradas_6b])[:, 1]

# O de fábrica e o afinado, medidos do MESMO jeito, com dois cortes:
#  - o do contrato: escolhido no simulado, no ponto em que erra o mesmo que a regra (é o que a prova vai usar);
#  - o do estudo: escolhido fora da dobra no estudo e levado ao simulado sem mexer (mede se a nota muda de escala).
_versoes = {"de fábrica": FABRICA_6B, "afinado": config_6b}
_linhas, apontas_11, notas_11, modelos_11 = [], {}, {}, {}
for _nome, _cfg in _versoes.items():
    modelos_11[_nome], notas_11[_nome] = treinar_6b(*_cfg)
    _corte_contrato = corte_com_precisao(_y_s, notas_11[_nome], precisao_regra)["corte"]
    _corte_estudo = corte_com_precisao(_y_e[_r3], notas_por_config_6b[_cfg][_r3], precisao_regra_estudo)["corte"]
    for _modo, _corte in [("corte do contrato (simulado)", _corte_contrato), ("corte do estudo", _corte_estudo)]:
        _aponta = notas_11[_nome] >= _corte
        apontas_11[(_nome, _modo)] = _aponta
        _p, _c = precisao_e_cobertura(_y_s, _aponta)
        _linhas.append({"versão": _nome, "corte": _modo, "C": _cfg[0], "peso dos duvidosos": _cfg[1],
                        "nota do texto": _cfg[2], "valor do corte": round(float(_corte), 4),
                        "acha de cada 10": round(10 * _c, 2), "precisão": round(_p, 3)})
conferencia_11 = pd.DataFrame(_linhas).set_index(["versão", "corte"])

# Cada linha contra o de fábrica com o corte do contrato, sorteando notícias do simulado.
_base = apontas_11[("de fábrica", "corte do contrato (simulado)")]
_dif, _faixa = [], []
for _chave in apontas_11:                       # a mesma ordem das linhas da tabela
    _iv = _iv11(_y_s, apontas_11[_chave], _base, simulado["grupo"])
    _dif.append(round(10 * _iv["diferenca"], 2))
    _faixa.append(f"{10 * _iv['de']:+.2f} a {10 * _iv['ate']:+.2f}")
conferencia_11["contra o de fábrica (de cada 10)"] = _dif
conferencia_11["faixa"] = _faixa

# O ensaio do contrato com o afinado (corte do contrato) contra a regra do app.
contrato_11 = passa_no_contrato(_y_s, apontas_11[("afinado", "corte do contrato (simulado)")], aponta_regra,
                                simulado["grupo"])
conferencia_11

In [ ]:
import json as _json11

# FIGURA DA ETAPA 11: quanto cada combinação acha dentro do estudo; uma linha por (peso, jeito da nota).
_fig, _ax = plt.subplots(figsize=(7.5, 4))
for _peso in PESOS_6B:
    for _jeito in JEITOS_6B:
        _g = grade_6b[(grade_6b["peso dos duvidosos"] == _peso) & (grade_6b["nota do texto"] == _jeito)]
        _ax.plot(_g["C"], _g["acha, de cada 10"], marker="o", linestyle="-" if _jeito == "como sai" else "--",
                 label=f"peso {_peso:g}, nota {_jeito}")
_ax.set_xscale("log")    # C vai de 0,003 a 10: em escala log, cada passo ocupa o mesmo espaço
_ax.axvline(FABRICA_6B[0], color="gray", linewidth=1, linestyle=":")
_ax.set_xlabel("C (à esquerda o modelo se segura mais; à direita, menos); pontilhado = fábrica")
_ax.set_ylabel("acha, de cada 10 (dentro do estudo)")
_ax.set_title("Ciclo 2, etapa 11: os botões do 6B, medidos dentro do estudo")
_ax.legend(fontsize=6, ncol=2)

_fab = conferencia_11.loc[("de fábrica", "corte do contrato (simulado)")]
_af = conferencia_11.loc[("afinado", "corte do contrato (simulado)")]
_af_est = conferencia_11.loc[("afinado", "corte do estudo")]
_mudou = config_6b != FABRICA_6B
_explicacao = (
    f"Dentro do estudo (rodada 3, errando o mesmo que a regra ali: precisão {precisao_regra_estudo:.1%}), a escolha "
    + (f"mudou para C={config_6b[0]:g}, peso {config_6b[1]:g} e nota {config_6b[2]}. " if _mudou
       else "ficou na posição de fábrica (nenhuma outra achou 0,1 de cada 10 a mais). ")
    + f"No simulado, com o corte do contrato: de fábrica acha {_fab['acha de cada 10']} de cada 10, o afinado "
    f"{_af['acha de cada 10']} ({_af['contra o de fábrica (de cada 10)']:+.2f}, faixa {_af['faixa']}). Com o corte "
    f"escolhido no estudo e levado ao simulado, o afinado acha {_af_est['acha de cada 10']} com precisão "
    f"{_af_est['precisão']:.1%}."
)
figura_11 = guard.fig(11, "botoes_do_6b", _fig, _explicacao, state_dir=PASTA_SAIDA)

# OS RESULTADOS EM ARQUIVO, para a sessão de IA ler do disco.
_pasta = PASTA_SAIDA / "resultados"
_pasta.mkdir(exist_ok=True)
(_pasta / "11_afinacao.json").write_text(_json11.dumps({
    "etapa": 11,
    "grade_C": GRADE_C_6B, "pesos": PESOS_6B, "jeitos": JEITOS_6B,
    "precisao_regra_estudo_r3": precisao_regra_estudo,
    "fabrica": list(FABRICA_6B), "escolhida": list(config_6b),
    "grade": grade_6b.to_dict(orient="records"),
    "conferencia_simulado": {" | ".join(k): v for k, v in conferencia_11.to_dict(orient="index").items()},
    "contrato_afinado_simulado": contrato_11,
}, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)) + "\n",
    encoding="utf-8", newline="\n")
mo.vstack([mo.image(src=str(figura_11), width=640), mo.md(_explicacao), conferencia_11])

# Ciclo 2, etapa 12: medir o escolhido com cuidado

**Em português:** até aqui medimos para escolher. Agora medimos o escolhido (o 6B com C=0,03) **por todos os
ângulos**, como uma revisão antes da prova: onde ele é forte, onde é frágil, e se dá para confiar na nota dele. Pelo
contrato, a nota de corte é escolhida no simulado, no ponto em que ele erra o mesmo que a regra do app; por isso os
números desta etapa saem um pouco otimistas. O número que vale sai da prova (etapa 14).

| O que olhamos | Em português | Em termos técnicos |
|---|---|---|
| **A cena das 100 fontes** | de 100 fontes do simulado, quantas o modelo e a regra acertam e erram, de cada tipo | matriz de confusão por 100 |
| **A curva do equilíbrio** | o que acontece se apertarmos ou afrouxarmos a nota de corte | curva precisão × cobertura |
| **As fatias do contrato** | onde a regra compara com cuidado; só relevância; sem os duvidosos; sem os "mesmo" de fonte velha | métricas por recorte, modelo e regra lado a lado |
| **A nota é confiável?** | nota 0,8 quer dizer "mesmo fato" 8 em cada 10 vezes? | calibração |
| **Quanto pesa cada informação** | o que o modelo mais usa para decidir | coeficientes da logística (entradas na mesma escala) |

**Já sabido da etapa 11 (não é expectativa):** no corte do contrato, o 6B acha 8,1 de cada 10 com precisão de 85,6%.

**O que eu espero, escrito antes de rodar:** onde a regra compara com cuidado, a precisão do 6B fica entre 85% e 95%
(a régua ficou em 88,9%; a regra, em 100%), ainda abaixo da regra; em "só relevância", o 6B acha de 7,5 a 8,5 de cada
10 com precisão de 80% a 88%; sem os duvidosos, acha de 0,2 a 0,6 a mais que no todo; sem os "mesmo" de fonte velha,
de 0 a 0,5 a mais. A nota sai otimista (o treino deu mais peso aos "mesmo"): nas notas acima de 0,75, a fração real
de "mesmo" fica entre 40% e 70%. Entre os pesos, o maior (em tamanho) é o da nota do texto, e o segundo é o da
semelhança do MPNet.

In [ ]:
import json as _json12
import hashlib as _hashlib12
import os as _os12

# O ESCOLHIDO: o 6B afinado da etapa 11, já treinado com o estudo inteiro, e a nota dele em cada fonte do simulado.
modelo_escolhido = modelos_11["afinado"]
nota_final = notas_11["afinado"]
# O CORTE do contrato: o mais frouxo em que ele erra no máximo o mesmo que a regra no simulado.
_c = corte_com_precisao(simulado["mesmo"], nota_final, piso=precisao_regra)
CORTE_FINAL = _c["corte"]
aponta_final = nota_final >= CORTE_FINAL
# Conferência: é o mesmo número da etapa 11 (se não for, alguma célula mudou no meio do caminho).
assert abs(10 * _c["cobertura"] - conferencia_11.loc[("afinado", "corte do contrato (simulado)"), "acha de cada 10"]) < 0.01

# O REGISTRO do escolhido, para a prova (etapa 14) usar EXATAMENTE este: configuração, corte, entradas, de onde vem a
# nota do texto e uma impressão digital das notas no simulado (se o modelo refeito der outras notas, aparece).
_pesos_ce = mo.notebook_dir().parent / "challenges" / "challenge-01-desinformacao-v2" / "data" / "interim" / "cross-encoder-ciclo-2" / "model.safetensors"
_registro = {
    "modelo": "6B: regressão logística (class_weight=balanced) sobre montar_preprocessador(com_regra=False, "
              "extras=('nota_texto',)); nota_texto = nota do cross-encoder BERTimbau (fora da dobra no estudo)",
    "C": config_6b[0], "peso_dos_duvidosos": config_6b[1], "nota_do_texto": config_6b[2],
    "corte": round(float(CORTE_FINAL), 6),
    "precisao_da_regra_no_simulado": round(float(precisao_regra), 6),
    "entradas": list(entradas_6b),
    "pesos_do_cross_encoder": "challenges/challenge-01-desinformacao-v2/data/interim/cross-encoder-ciclo-2 (fora do git)",
    "tamanho_do_model_safetensors": _pesos_ce.stat().st_size if _pesos_ce.is_file() else None,
    "impressao_digital_das_notas_no_simulado": _hashlib12.sha256(np.round(nota_final, 6).tobytes()).hexdigest(),
    "escolhido_em": "etapa 10 (o 6B, pelo Matheus), etapa 11 (botões, dentro do estudo), etapa 12 (corte, no simulado)",
}
if not int(_os12.environ.get("ML_AMOSTRA_TESTE", "0")):     # o teste pequeno nunca grava este registro
    (mo.notebook_dir() / "modelo_escolhido.json").write_text(
        _json12.dumps(_registro, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
_p, _cob = precisao_e_cobertura(simulado["mesmo"], aponta_final)
{"C": config_6b[0], "corte": round(float(CORTE_FINAL), 4), "acha, de cada 10": round(10 * _cob, 2),
 "precisão": round(_p, 3), "regra apontou": int(aponta_regra.sum()), "6B apontou": int(aponta_final.sum())}

In [ ]:
# FIGURA 12A: A CENA DAS 100 FONTES (a matriz de confusão contada por 100 fontes do simulado), do 6B e da regra.
_y = simulado["mesmo"].to_numpy()
_n = len(_y)

def _cena(aponta):
    _por_100 = lambda m: 100 * m.sum() / _n        # quantas, a cada 100 fontes
    return np.array([[_por_100(_y & aponta), _por_100(_y & ~aponta)],       # é o mesmo fato: apontou / deixou passar
                     [_por_100(~_y & aponta), _por_100(~_y & ~aponta)]])    # não é: apontou por engano / ficou quieto

_fig, _eixos = plt.subplots(1, 2, figsize=(9, 3.8))
for _ax, (_nome, _aponta) in zip(_eixos, [("6B (o escolhido)", aponta_final), ("regra do app", aponta_regra)]):
    _m = _cena(_aponta)
    _ax.imshow(_m, cmap="Blues", vmin=0, vmax=100)
    for (_i, _j), _v in np.ndenumerate(_m):
        _ax.text(_j, _i, f"{_v:.1f}", ha="center", va="center", fontsize=13)
    _ax.set_xticks([0, 1], ["aponta\n\"mesmo fato\"", "não aponta"])
    _ax.set_yticks([0, 1], ["é o mesmo fato", "não é"])
    _ax.set_title(_nome)
_fig.suptitle("De cada 100 fontes do simulado")
_m6, _mr = _cena(aponta_final), _cena(aponta_regra)
_explicacao = (
    f"De cada 100 fontes do simulado, {_m6[0].sum():.1f} são o mesmo fato. O 6B aponta {_m6[0, 0]:.1f} delas e deixa "
    f"{_m6[0, 1]:.1f} passar; a regra aponta {_mr[0, 0]:.1f} e deixa {_mr[0, 1]:.1f}. Por engano, o 6B aponta "
    f"{_m6[1, 0]:.1f} e a regra {_mr[1, 0]:.1f}: de tudo o que mostra, o 6B erra {_m6[1, 0]:.1f} em "
    f"{_m6[:, 0].sum():.1f} e a regra {_mr[1, 0]:.1f} em {_mr[:, 0].sum():.1f}, a mesma proporção (é o contrato)."
)
figura_cena = guard.fig(12, "cena_das_100_fontes", _fig, _explicacao, state_dir=PASTA_SAIDA)
mo.vstack([mo.image(src=str(figura_cena), width=700), mo.md(_explicacao)])

In [ ]:
from sklearn.metrics import average_precision_score as _ap12, precision_recall_curve as _prc12

# FIGURA 12B: a curva do equilíbrio do 6B, com a régua (etapa 9), a regra do app e o ponto escolhido.
_y = simulado["mesmo"].to_numpy()
_fig, _ax = plt.subplots(figsize=(6.5, 4.2))
for _nome, _nota in [("6B, o escolhido (todos os cortes)", nota_final), ("régua da etapa 9", nota_logistica)]:
    _prec, _cob, _ = _prc12(_y, _nota)
    _ax.plot(_cob, _prec, label=_nome)
_pf, _cf = precisao_e_cobertura(_y, aponta_final)
_pr, _cr = precisao_e_cobertura(_y, aponta_regra)
_ax.axhline(_pr - MARGEM, linestyle="--", color="gray", linewidth=1, label="precisão da regra − 3 pontos (limite)")
_ax.scatter([_cr], [_pr], color="black", zorder=3, label="regra do app")
_ax.scatter([_cf], [_pf], color="red", zorder=3, label="6B, no corte do contrato")
_ax.set_xlabel("de cada 10 fontes do mesmo fato, quantas acha (cobertura)")
_ax.set_ylabel("do que mostra, quanto está certo (precisão)")
_ax.set_ylim(0.4, 1.02)
_ax.legend(fontsize=7, loc="lower left")
_ax.set_title("Ciclo 2, etapa 12: o equilíbrio entre achar e errar")
contrato_12 = passa_no_contrato(_y, aponta_final, aponta_regra, simulado["grupo"])
_explicacao = (
    f"Afrouxar o corte acha mais e erra mais; apertar acha menos e erra menos. No corte do contrato (nota "
    f"{CORTE_FINAL:.3f}), o 6B acha {10 * _cf:.1f} de cada 10 com precisão de {_pf:.1%}; a regra acha {10 * _cr:.1f} "
    f"com {_pr:.1%}. Diferença: {10 * contrato_12['cobertura']['diferenca']:+.2f} de cada 10 (faixa de "
    f"{10 * contrato_12['cobertura']['de']:+.2f} a {10 * contrato_12['cobertura']['ate']:+.2f}, sorteando notícias). "
    f"PR-AUC do 6B {_ap12(_y, nota_final):.3f}; da régua {_ap12(_y, nota_logistica):.3f}."
)
figura_curva = guard.fig(12, "curva_do_equilibrio", _fig, _explicacao, state_dir=PASTA_SAIDA)

# AS FATIAS DO CONTRATO: o 6B e a regra lado a lado (e a régua, para comparar), em cada recorte.
_fatias = {
    "todo o simulado": simulado.index == simulado.index,
    "a regra comparou com cuidado": (simulado["comparou"] == 1).to_numpy(),
    "só relevância": (simulado["comparou"] == 0).to_numpy(),
    "sem os duvidosos (maioria de três)": (simulado["origem"] != "maioria de três").to_numpy(),
    "sem os \"mesmo\" de fonte velha": ~simulado["mesmo_de_fonte_velha"].to_numpy(dtype=bool),
}
_linhas = []
for _nome, _m in _fatias.items():
    _p6, _c6 = precisao_e_cobertura(_y[_m], aponta_final[_m])
    _prg, _crg = precisao_e_cobertura(_y[_m], aponta_regra[_m])
    _plg, _clg = precisao_e_cobertura(_y[_m], aponta_logistica[_m])
    _linhas.append({"fatia": _nome, "fontes": int(_m.sum()), "do mesmo fato": int(_y[_m].sum()),
                    "6B: acha de cada 10": round(10 * _c6, 2), "6B: precisão": round(_p6, 3),
                    "regra: acha de cada 10": round(10 * _crg, 2), "regra: precisão": round(_prg, 3),
                    "régua: acha de cada 10": round(10 * _clg, 2), "régua: precisão": round(_plg, 3)})
por_fatia_12 = pd.DataFrame(_linhas).set_index("fatia")
mo.vstack([mo.image(src=str(figura_curva), width=560), mo.md(_explicacao), por_fatia_12])

In [ ]:
# FIGURA 12C: A NOTA É CONFIÁVEL? Junta as fontes por faixa de nota (0 a 0,1; 0,1 a 0,2; ...) e conta, em cada
# faixa, quantas eram mesmo o mesmo fato. Se a nota fosse uma "chance" honesta, os pontos cairiam na diagonal.
_y = simulado["mesmo"].to_numpy()
_faixas = np.linspace(0, 1, 11)                          # 11 bordas = 10 faixas
_qual = np.clip(np.digitize(nota_final, _faixas) - 1, 0, 9)
calibracao_12 = []
for _k in range(10):
    _dentro = _qual == _k
    if _dentro.sum() >= 5:                                # faixa com menos de 5 fontes não diz nada
        calibracao_12.append({"nota média": float(nota_final[_dentro].mean()),
                              "fração real": float(_y[_dentro].mean()), "fontes": int(_dentro.sum())})
_fig, _ax = plt.subplots(figsize=(5, 4.2))
_ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="nota honesta (diagonal)")
_ax.plot([c["nota média"] for c in calibracao_12], [c["fração real"] for c in calibracao_12], marker="o",
         label="6B, o escolhido")
_ax.set_xlabel("nota média que o modelo deu")
_ax.set_ylabel("fração que era mesmo o mesmo fato")
_ax.set_title("A nota do 6B é uma chance honesta?")
_ax.legend(fontsize=7)
_alto = [c for c in calibracao_12 if c["nota média"] >= 0.75]
_explicacao = (
    "Cada ponto junta as fontes de uma faixa de nota. Pontos ABAIXO da diagonal: a nota é otimista (o modelo diz 0,8 e "
    "a fração real é menor). Nas notas a partir de 0,75, a fração real de \"mesmo fato\" ficou em "
    + (", ".join(f"{c['fração real']:.0%} (nota {c['nota média']:.2f}, {c['fontes']} fontes)" for c in _alto)
       if _alto else "faixa sem fontes suficientes")
    + ". A decisão não sofre (o corte foi escolhido olhando o resultado real), mas a nota só pode ser mostrada ao "
    "leitor como \"chance\" depois de um ajuste."
)
figura_calibracao = guard.fig(12, "a_nota_e_confiavel", _fig, _explicacao, state_dir=PASTA_SAIDA)
mo.vstack([mo.image(src=str(figura_calibracao), width=420), mo.md(_explicacao)])

In [ ]:
import json as _json12b

# QUANTO PESA CADA INFORMAÇÃO: como todas as entradas foram postas na mesma escala (etapa 8), o tamanho do peso da
# logística compara a força de cada uma. Positivo puxa para "mesmo fato"; negativo, para "não é".
_logistica = modelo_escolhido.named_steps["logistica"]
try:
    _nomes = list(modelo_escolhido.named_steps["preproc"].get_feature_names_out())
except Exception:                                  # se alguma peça não souber dar nome, numera
    _nomes = [f"coluna {i}" for i in range(_logistica.coef_.shape[1])]
pesos_12 = (pd.DataFrame({"informação": _nomes, "peso": _logistica.coef_[0]})
            .assign(tamanho=lambda d: d["peso"].abs())
            .sort_values("tamanho", ascending=False, kind="stable")
            .drop(columns="tamanho").reset_index(drop=True))

# OS RESULTADOS EM ARQUIVO, para a sessão de IA ler do disco.
_pasta = PASTA_SAIDA / "resultados"
_pasta.mkdir(exist_ok=True)
(_pasta / "12_avaliacao.json").write_text(_json12b.dumps({
    "etapa": 12, "configuracao": list(config_6b), "corte": float(CORTE_FINAL),
    "contrato_no_simulado": contrato_12,
    "fatias": por_fatia_12.to_dict(orient="index"),
    "calibracao": calibracao_12,
    "pesos": pesos_12.round(4).to_dict(orient="records"),
}, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)) + "\n",
    encoding="utf-8", newline="\n")
mo.vstack([mo.md("**Quanto pesa cada informação** (as 12 mais fortes; a lista inteira está no arquivo de resultados)"),
           pesos_12.head(12).round(3)])